In [1]:
#!/usr/bin/env python3
"""
================================================================================
ISRO PS-4: Route Resilience — Occlusion-Robust Road Extraction & Graph Analysis
================================================================================
• 100% Self-Contained: Paste directly into a single Kaggle Notebook cell & run.
• Fixes `FileNotFoundError` (no external .py file required on disk).
• Supports both full DeepGlobe training AND small/unlabelled Sample Data training.
• Fixes 96 missing checkpoint keys (420/420 layer match) & 3% IoU collapse.
• Automatically saves `/kaggle/working/route_resilience_master.ipynb` at the end.
================================================================================
"""

# ==============================================================================
# 1. ENVIRONMENT SETUP, THREAD SAFETY & HARDWARE DETECTION
# ==============================================================================
import os
import sys
import gc
import glob
import math
import time
import json
import random
import inspect
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from contextlib import nullcontext

os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONHASHSEED'] = '42'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

warnings.filterwarnings('ignore')

import numpy as np
import cv2
cv2.setNumThreads(0)

import networkx as nx
from scipy.ndimage import distance_transform_edt

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# NumPy 1.x / 2.x compatible trapezoidal integration
trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"✅ NVIDIA GPU Initialized: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}")
else:
    print(f"ℹ️ Running on CPU | PyTorch {torch.__version__}")


✅ NVIDIA GPU Initialized: Tesla T4 | PyTorch 2.11.0+cu128


In [2]:
# ==============================================================================
# CELL 2: Automatic Path Discovery & Unified Configuration (TARGETS EXACT FILENAME)
# ==============================================================================
class KagglePathFinder:
    @staticmethod
    def discover() -> Dict[str, Optional[str]]:
        paths = {
            'weights': None, 'train_dir': None, 'sample_dir': None,
            'output_dir': '/kaggle/working/output' if os.path.exists('/kaggle') else './output'
        }
        os.makedirs(paths['output_dir'], exist_ok=True)

        # 1. Explicitly target the exact folder "0.70_iou_1" and filename "0.70iou_1"
        # The asterisk (*) ensures it catches it whether it has a .pth extension or not.
        weight_candidates = [
            '/kaggle/input/0.70_iou_1/0.70iou_1*',
            '/kaggle/input/0-70-iou-1/0.70iou_1*',
            '/kaggle/input/*/0.70iou_1*',
            '/kaggle/input/datasets/*/0.70_iou_1/0.70iou_1*',
            '/kaggle/input/datasets/*/0-70-iou-1/0.70iou_1*',
            '/kaggle/working/output/best_model.pth'
        ]
        for pattern in weight_candidates:
            matches = sorted(glob.glob(pattern))
            if matches:
                paths['weights'] = matches[0]
                break

        # Fallback recursive scan across /kaggle/input targeting the exact filename
        if not paths['weights'] and os.path.exists('/kaggle/input'):
            exact_files = sorted(glob.glob('/kaggle/input/**/0.70iou_1*', recursive=True))
            if exact_files:
                paths['weights'] = exact_files[0]
            else:
                # Generic fallback to any .pth with '70' in the name
                all_pths = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
                if all_pths:
                    preferred = [p for p in all_pths if '70' in p]
                    paths['weights'] = preferred[0] if preferred else all_pths[0]

        # 2. Discover DeepGlobe Train Directory
        train_candidates = [
            '/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train',
            '/kaggle/input/deepglobe-road-extraction-dataset/train',
            './data/train'
        ]
        for tc in train_candidates:
            if os.path.isdir(tc):
                paths['train_dir'] = tc
                break
        if not paths['train_dir'] and os.path.exists('/kaggle/input'):
            mask_matches = glob.glob('/kaggle/input/**/*_mask.png', recursive=True)
            if mask_matches:
                paths['train_dir'] = str(Path(mask_matches[0]).parent)

        # 3. Discover Sample Imagery Directory
        sample_candidates = [
            '/kaggle/input/datasets/rajitdhakad/sample-data-bah',
            '/kaggle/input/sample-data-bah',
            '/kaggle/input/sample-data-bah/my_images',
            './sample_images'
        ]
        for sc in sample_candidates:
            if os.path.isdir(sc):
                paths['sample_dir'] = sc
                break

        if not paths['sample_dir']:
            paths['sample_dir'] = paths['train_dir']

        print("🔍 Discovered Paths:")
        for k, v in paths.items():
            print(f"  • {k:<12}: {v}")

        if paths['weights'] is None:
            raise FileNotFoundError(
                "🚨 FATAL: Could not find the weight file '0.70iou_1' in /kaggle/input! "
                "Verify the dataset is attached in the right-hand panel."
            )

        return paths

CONFIG_PATHS = KagglePathFinder.discover()

class PipelineConfig:
    IMAGE_SIZE: int = 512
    IN_CHANNELS: int = 3
    NUM_ORIENT_BINS: int = 37

    BATCH_SIZE: int = 4
    GRAD_ACCUM_STEPS: int = 2
    EPOCHS: int = 4                # Short 4-epoch tuning run
    BASE_LR: float = 5.0e-5        # Lowered LR for fine-tuning plateau
    BACKBONE_LR: float = 1.0e-5    
    WEIGHT_DECAY: float = 1e-4
    ADAM_EPS: float = 1e-4
    AMP_ENABLED: bool = True
    MAX_GRAD_NORM: float = 1.0
    USE_TTA: bool = True

    VAL_SPLIT: float = 0.10
    SEED: int = 42
    NUM_WORKERS: int = 0

    # LATE-STAGE LOSS SHIFT: Less focus on Dice area, higher focus on clDice topology
    W_FOCAL: float = 1.0
    W_DICE: float = 0.60
    W_CLDICE_INIT: float = 0.25
    W_CLDICE_FINAL: float = 0.45
    W_BOUNDARY: float = 0.20
    W_OCC: float = 0.15
    W_ORIENT: float = 0.10

    # TUNED DSU-MST GRAPH HEALING BOUNDS
    MIN_COMPONENT_AREA: int = 35
    MAX_GAP_PX: float = 55.0
    MAX_BEARING_ANGLE_DEG: float = 55.0
    METERS_PER_PIXEL: float = 5.0
    VEHICLE_SPEED_MS: float = 8.333
    BETWEENNESS_K: int = 300
    FLOOD_RADIUS_PX: float = 45.0
    RELAXED_IOU_BUFFER_PX: int = 4

CFG = PipelineConfig()

🔍 Discovered Paths:
  • weights     : /kaggle/input/datasets/rajitdhakad/0-70-iou-1/0.70iou_1
  • train_dir   : /kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train
  • sample_dir  : /kaggle/input/datasets/rajitdhakad/sample-data-bah
  • output_dir  : /kaggle/working/output


In [3]:
# ==============================================================================
# 3. INDIAN ATMOSPHERIC OCCLUSION SYNTHESIS & ORIENTATION TARGETS
# ==============================================================================
def compute_road_orientations(mask_hw_uint8: np.ndarray, num_bins: int = 36) -> np.ndarray:
    road_mask = (mask_hw_uint8 > 127).astype(np.uint8)
    orient_hw = np.zeros(mask_hw_uint8.shape, dtype=np.int64)
    if not np.any(road_mask):
        return orient_hw

    blurred = cv2.GaussianBlur(road_mask.astype(np.float32), (5, 5), sigmaX=1.0)
    sobel_x = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    angle = np.mod(np.arctan2(-sobel_y, sobel_x), np.pi)
    bin_idx = np.floor(angle / (np.pi / float(num_bins))).astype(np.int64)
    bin_idx = np.clip(bin_idx, 0, num_bins - 1) + 1
    orient_hw[road_mask > 0] = bin_idx[road_mask > 0]
    return orient_hw

def apply_indian_occlusions(rgb_uint8_hwc: np.ndarray, mask_hw_uint8: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    h, w, _ = rgb_uint8_hwc.shape
    aug_rgb = rgb_uint8_hwc.copy().astype(np.float32)
    occ_mask = np.zeros((h, w), dtype=np.uint8)
    road_coords = np.argwhere(mask_hw_uint8 > 127)

    num_shadows = random.randint(3, 6)
    for _ in range(num_shadows):
        if len(road_coords) > 0:
            cy, cx = road_coords[random.randint(0, len(road_coords) - 1)]
            cx = int(np.clip(cx + random.randint(-12, 12), 0, w - 1))
            cy = int(np.clip(cy + random.randint(-12, 12), 0, h - 1))
        else:
            cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)

        axes = (random.randint(18, 55), random.randint(10, 30))
        angle = random.randint(0, 180)
        ellipse_pattern = np.zeros((h, w), dtype=np.float32)
        cv2.ellipse(ellipse_pattern, (cx, cy), axes, angle, 0, 360, 1.0, -1)
        blurred_ellipse = cv2.GaussianBlur(ellipse_pattern, (21, 21), sigmaX=7.0)
        shadow_darkening = random.uniform(0.45, 0.75)
        for c in range(3):
            aug_rgb[:, :, c] = aug_rgb[:, :, c] * (1.0 - blurred_ellipse * shadow_darkening)
        occ_mask[blurred_ellipse > 0.35] = 255

    if random.random() < 0.30:
        cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
        axes = (random.randint(40, 110), random.randint(30, 80))
        angle = random.randint(0, 180)
        cloud_pattern = np.zeros((h, w), dtype=np.float32)
        cv2.ellipse(cloud_pattern, (cx, cy), axes, angle, 0, 360, 1.0, -1)
        cloud_blurred = cv2.GaussianBlur(cloud_pattern, (35, 35), sigmaX=12.0)
        for c in range(3):
            aug_rgb[:, :, c] = aug_rgb[:, :, c] * (1.0 - cloud_blurred * 0.45) + 235.0 * (cloud_blurred * 0.45)
        occ_mask[cloud_blurred > 0.30] = 255

    return np.clip(aug_rgb, 0.0, 255.0).astype(np.uint8), occ_mask

In [4]:

# ==============================================================================
# 4. ARCHITECTURE & SMART 420/420 LAYER CHECKPOINT LOADER
# ==============================================================================
class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int, patch_size: int, stride: int, padding: int):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=stride, padding=padding)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor):
        x = self.proj(x)
        b, c, h, w = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, (h, w)

class EfficientSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, sr_ratio: int = 1):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.sr_ratio = sr_ratio

        self.q = nn.Linear(dim, dim, bias=True)
        self.kv = nn.Linear(dim, dim * 2, bias=True)
        self.proj = nn.Linear(dim, dim)

        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr = None
            self.norm = None

    def forward(self, x: torch.Tensor, h: int, w: int):
        b, n, c = x.shape
        q = self.q(x).reshape(b, n, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        if self.sr is not None:
            x_spatial = x.permute(0, 2, 1).reshape(b, c, h, w)
            x_reduced = self.sr(x_spatial).reshape(b, c, -1).permute(0, 2, 1)
            x_reduced = self.norm(x_reduced)
            kv = self.kv(x_reduced).reshape(b, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(b, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)

        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(b, n, c)
        return self.proj(out)

class MixFFN(nn.Module):
    def __init__(self, in_features: int, hidden_features: int):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, stride=1, padding=1, groups=hidden_features, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)

    def forward(self, x: torch.Tensor, h: int, w: int):
        b, n, c = x.shape
        x = self.fc1(x)
        x = x.transpose(1, 2).view(b, -1, h, w)
        x = self.dwconv(x)
        x = self.act(x)
        x = x.flatten(2).transpose(1, 2)
        return self.fc2(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, sr_ratio: int = 1, mlp_ratio: float = 4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = EfficientSelfAttention(dim, num_heads=num_heads, sr_ratio=sr_ratio)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = MixFFN(dim, hidden_features=int(dim * mlp_ratio))

    def forward(self, x: torch.Tensor, h: int, w: int):
        x = x + self.attn(self.norm1(x), h, w)
        x = x + self.ffn(self.norm2(x), h, w)
        return x

class MiTB2Backbone(nn.Module):
    def __init__(self, in_channels: int = 3):
        super().__init__()
        self.patch_embed1 = OverlapPatchEmbed(in_channels, 64, 7, 4, 3)
        self.block1 = nn.ModuleList([TransformerBlock(64, 1, 8) for _ in range(3)])
        self.norm1 = nn.LayerNorm(64)

        self.patch_embed2 = OverlapPatchEmbed(64, 128, 3, 2, 1)
        self.block2 = nn.ModuleList([TransformerBlock(128, 2, 4) for _ in range(4)])
        self.norm2 = nn.LayerNorm(128)

        self.patch_embed3 = OverlapPatchEmbed(128, 320, 3, 2, 1)
        self.block3 = nn.ModuleList([TransformerBlock(320, 5, 2) for _ in range(6)])
        self.norm3 = nn.LayerNorm(320)

        self.patch_embed4 = OverlapPatchEmbed(320, 512, 3, 2, 1)
        self.block4 = nn.ModuleList([TransformerBlock(512, 8, 1) for _ in range(3)])
        self.norm4 = nn.LayerNorm(512)

    def forward(self, x: torch.Tensor):
        b = x.shape[0]
        x, (h1, w1) = self.patch_embed1(x)
        for blk in self.block1: x = blk(x, h1, w1)
        e1 = self.norm1(x).transpose(1, 2).view(b, 64, h1, w1)

        x, (h2, w2) = self.patch_embed2(e1)
        for blk in self.block2: x = blk(x, h2, w2)
        e2 = self.norm2(x).transpose(1, 2).view(b, 128, h2, w2)

        x, (h3, w3) = self.patch_embed3(e2)
        for blk in self.block3: x = blk(x, h3, w3)
        e3 = self.norm3(x).transpose(1, 2).view(b, 320, h3, w3)

        x, (h4, w4) = self.patch_embed4(e3)
        for blk in self.block4: x = blk(x, h4, w4)
        e4 = self.norm4(x).transpose(1, 2).view(b, 512, h4, w4)
        return [e1, e2, e3, e4]

class MiTB2Encoder(nn.Module):
    def __init__(self, in_channels: int = 3):
        super().__init__()
        self.backbone = MiTB2Backbone(in_channels=in_channels)

    def forward(self, x):
        return self.backbone(x)

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))

class DoubleDSCBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = DepthwiseSeparableConv(in_ch, out_ch)
        self.c2 = DepthwiseSeparableConv(out_ch, out_ch)

    def forward(self, x):
        return self.c2(self.c1(x))

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        bot = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, bot, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(bot, channels, 1, bias=False)
        )

    def forward(self, x):
        avg_pool = F.adaptive_avg_pool2d(x, 1)
        max_pool = F.adaptive_max_pool2d(x, 1)
        w = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * w

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg_p = x.mean(dim=1, keepdim=True)
        max_p, _ = x.max(dim=1, keepdim=True)
        w = torch.sigmoid(self.conv(torch.cat([avg_p, max_p], dim=1)))
        return x * w

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ch = ChannelAttention(channels, reduction)
        self.sp = SpatialAttention()

    def forward(self, x):
        return self.sp(self.ch(x))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.block = DoubleDSCBlock(in_ch + skip_ch, out_ch)
        self.cbam = CBAM(out_ch)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.block(x))

class Decoder(nn.Module):
    def __init__(self, enc_ch=(64, 128, 320, 512)):
        super().__init__()
        e1, e2, e3, e4 = enc_ch
        self.dec4 = DecoderBlock(e4, e3, 256)
        self.dec3 = DecoderBlock(256, e2, 128)
        self.dec2 = DecoderBlock(128, e1, 64)
        self.dec1 = DecoderBlock(64, 0, 32)
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, feats):
        e1, e2, e3, e4 = feats
        d4 = self.dec4(e4, skip=e3)
        d3 = self.dec3(d4, skip=e2)
        d2 = self.dec2(d3, skip=e1)
        d1 = self.dec1(d2, skip=None)
        return self.final_up(d1)

class RoadExtractionModelV3(nn.Module):
    def __init__(self, in_channels: int = 3):
        super().__init__()
        self.in_channels = in_channels
        self.encoder = MiTB2Encoder(in_channels=in_channels)
        self.decoder = Decoder()
        self.road_head = nn.Conv2d(32, 1, kernel_size=1)
        self.occlusion_head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )
        self.orientation_head = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 37, 1)
        )

    def forward(self, x: torch.Tensor):
        if x.shape[1] > self.in_channels:
            x = x[:, :self.in_channels]
        enc_feats = self.encoder(x)
        dec_feats = self.decoder(enc_feats)
        occl_logits = self.occlusion_head(dec_feats)
        gate = torch.sigmoid(occl_logits.detach())
        gated = dec_feats * (1.0 + 0.5 * gate)
        road_logits = self.road_head(gated)
        orient_logits = self.orientation_head(dec_feats)
        return road_logits, occl_logits, orient_logits

def smart_load_checkpoint(model: nn.Module, pretrained_path: Optional[str]) -> Tuple[nn.Module, Optional[dict]]:
    """3-Pass Checkpoint Loader that resolves the 96 renamed keys in best_model.pth."""
    if not pretrained_path or not os.path.exists(pretrained_path):
        print("ℹ️ No pretrained checkpoint found. Initializing fresh model.")
        return model.to(DEVICE), None

    print(f"📦 Loading checkpoint from: {pretrained_path}")
    try:
        ckpt = torch.load(pretrained_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(pretrained_path, map_location='cpu')

    sd = ckpt.get('model_state_dict', ckpt.get('full_state_dict', ckpt.get('state_dict', ckpt)))
    model_sd = model.state_dict()
    new_sd = {}
    used_ckpt_keys = set()

    # Pass 1: Exact name + shape match (or 5ch -> 3ch first conv slice)
    for k, m_val in model_sd.items():
        if k in sd:
            c_val = sd[k]
            if c_val.shape == m_val.shape:
                new_sd[k] = c_val
                used_ckpt_keys.add(k)
            elif (c_val.ndim == 4 and m_val.ndim == 4 and
                  c_val.shape[0] == m_val.shape[0] and c_val.shape[2:] == m_val.shape[2:] and
                  c_val.shape[1] == 5 and m_val.shape[1] == 3):
                new_sd[k] = c_val[:, :3, :, :].clone()
                used_ckpt_keys.add(k)

    exact_count = len(new_sd)

    # Pass 2: Suffix similarity + exact shape match (handles `decoder.dec4` vs `dec4`, etc.)
    missing_keys = [k for k in model_sd.keys() if k not in new_sd]
    unexpected_keys = [k for k in sd.keys() if k not in used_ckpt_keys]

    for m_key in list(missing_keys):
        m_shape = model_sd[m_key].shape
        m_tail = ".".join(m_key.split(".")[-2:])
        for u_key in list(unexpected_keys):
            u_tail = ".".join(u_key.split(".")[-2:])
            if sd[u_key].shape == m_shape and m_tail == u_tail:
                new_sd[m_key] = sd[u_key]
                used_ckpt_keys.add(u_key)
                missing_keys.remove(m_key)
                unexpected_keys.remove(u_key)
                break

    # Pass 3: Sequential order + exact shape & dtype match for any remaining keys
    for m_key in list(missing_keys):
        m_val = model_sd[m_key]
        for u_key in list(unexpected_keys):
            u_val = sd[u_key]
            if u_val.shape == m_val.shape and u_val.dtype == m_val.dtype:
                new_sd[m_key] = u_val
                used_ckpt_keys.add(u_key)
                missing_keys.remove(m_key)
                unexpected_keys.remove(u_key)
                break

    model.load_state_dict(new_sd, strict=False)
    print(f"✅ Smart Checkpoint Loaded: {len(new_sd)}/{len(model_sd)} layers matched! (Exact: {exact_count}, Remapped: {len(new_sd) - exact_count})")
    return model.to(DEVICE), ckpt




In [5]:
# ==============================================================================
# 5. DATASET & DATALOADER (SUPPORTS BOTH DEEPGLOBE & UNLABELLED SAMPLE DATA)
# ==============================================================================
class DeepGlobeJPGDataset(Dataset):
    def __init__(self, image_paths: List[str], mask_paths: List[str],
                 img_size: int = 512, is_training: bool = True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.is_training = is_training
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        if self.mask_paths and idx < len(self.mask_paths) and self.mask_paths[idx]:
            m = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
            if m is not None:
                mask = m

        if img.shape[0] != self.img_size or img.shape[1] != self.img_size:
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        if mask.shape[0] != self.img_size or mask.shape[1] != self.img_size:
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        occ_mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        if self.is_training:
            img, occ_mask = apply_indian_occlusions(img, mask)
            if random.random() < 0.5:
                img, mask, occ_mask = np.fliplr(img).copy(), np.fliplr(mask).copy(), np.fliplr(occ_mask).copy()
            if random.random() < 0.5:
                img, mask, occ_mask = np.flipud(img).copy(), np.flipud(mask).copy(), np.flipud(occ_mask).copy()
            if random.random() < 0.5:
                k = random.choice([1, 2, 3])
                img, mask, occ_mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy(), np.rot90(occ_mask, k).copy()
        else:
            rng_state = random.getstate()
            random.seed(idx + 1000)
            img, occ_mask = apply_indian_occlusions(img, mask)
            random.setstate(rng_state)

        rgb_norm = (img.astype(np.float32) / 255.0 - self.mean) / self.std
        img_tensor = torch.from_numpy(rgb_norm.transpose(2, 0, 1).copy()).float()

        road_bin = (mask > 127)
        road_mask_tensor = torch.from_numpy(road_bin.astype(np.float32)).unsqueeze(0)
        occ_mask_tensor = torch.from_numpy((occ_mask > 127).astype(np.float32)).unsqueeze(0)

        dt = distance_transform_edt(road_bin)
        thin_weight = ((dt > 0) & (dt <= 3)).astype(np.float32) * 3.0 + 1.0
        occ_road_boost = (road_bin & (occ_mask > 127)).astype(np.float32) * 2.0
        pixel_weight_tensor = torch.from_numpy(thin_weight + occ_road_boost).unsqueeze(0).float()

        orient_arr = compute_road_orientations(mask, num_bins=36)
        orient_tensor = torch.from_numpy(orient_arr).long()

        return img_tensor, road_mask_tensor, occ_mask_tensor, orient_tensor, pixel_weight_tensor

def generate_pseudo_masks_for_sample_data(image_paths: List[str], model: Optional[nn.Module], out_mask_dir: str) -> List[str]:
    """Generates high-confidence pseudo-masks when training on unlabelled sample .jpg folders."""
    os.makedirs(out_mask_dir, exist_ok=True)
    mask_paths = []
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    if model is not None:
        model.eval()

    print(f"🛠️ Generating pseudo-masks for {len(image_paths)} unlabelled sample image(s)...")
    for p in image_paths:
        stem = Path(p).stem
        m_path = os.path.join(out_mask_dir, f"{stem}_mask.png")
        if os.path.exists(m_path):
            mask_paths.append(m_path)
            continue

        img_bgr = cv2.imread(p)
        if img_bgr is None:
            cv2.imwrite(m_path, np.zeros((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), dtype=np.uint8))
            mask_paths.append(m_path)
            continue

        img_512 = cv2.resize(img_bgr, (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE))
        mask_u8 = None
        if model is not None:
            img_rgb = cv2.cvtColor(img_512, cv2.COLOR_BGR2RGB)
            rgb_n = (img_rgb.astype(np.float32) / 255.0 - mean) / std
            inp = torch.from_numpy(rgb_n.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                r_log, _, _ = model(inp)
                prob = torch.sigmoid(r_log).squeeze().cpu().numpy()
            thresh = 0.50 if (prob > 0.50).sum() > 200 else float(np.percentile(prob, 92))
            mask_u8 = (prob > thresh).astype(np.uint8) * 255

        if mask_u8 is None or mask_u8.sum() == 0:
            gray = cv2.cvtColor(img_512, cv2.COLOR_BGR2GRAY)
            edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 15, -5)
            mask_u8 = cv2.morphologyEx(edges, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

        cv2.imwrite(m_path, mask_u8)
        mask_paths.append(m_path)
    return mask_paths

def build_dataloaders(data_dir: str, batch_size: int = 4, val_split: float = 0.10,
                      seed: int = 42, pretrained_model: Optional[nn.Module] = None):
    sat_files = sorted(glob.glob(os.path.join(data_dir, '**/*_sat.jpg'), recursive=True))
    if not sat_files:
        sat_files = sorted([
            p for p in glob.glob(os.path.join(data_dir, '**/*.jpg'), recursive=True)
            if not p.endswith('_mask.jpg')
        ])

    image_paths, mask_paths = [], []
    for p in sat_files:
        stem = Path(p).stem.replace('_sat', '')
        parent = str(Path(p).parent)
        candidates = [
            p.replace('_sat.jpg', '_mask.png'),
            p.replace('.jpg', '_mask.png'),
            p.replace('_sat.jpg', '_mask.jpg'),
            os.path.join(parent, f"{stem}_mask.png"),
            os.path.join(parent, 'masks', f"{stem}.png"),
            os.path.join(parent, 'masks', f"{stem}_mask.png"),
            os.path.join(str(Path(parent).parent), 'masks', f"{stem}.png"),
        ]
        found_mask = next((c for c in candidates if os.path.exists(c) and c != p), None)
        if found_mask:
            image_paths.append(p)
            mask_paths.append(found_mask)

    # Fallback: If user is training on unlabelled sample .jpg data without masks
    if not image_paths and sat_files:
        print(f"ℹ️ No ground-truth masks found in '{data_dir}'. Activating Pseudo-Mask Generator for sample data training!")
        image_paths = sat_files
        pseudo_dir = os.path.join(CONFIG_PATHS['output_dir'], 'pseudo_masks')
        mask_paths = generate_pseudo_masks_for_sample_data(image_paths, pretrained_model, pseudo_dir)

    if not image_paths:
        # Final fallback: create 4 synthetic satellite road pairs so training never crashes
        synth_dir = os.path.join(CONFIG_PATHS['output_dir'], 'synth_train')
        os.makedirs(synth_dir, exist_ok=True)
        for i in range(4):
            sp = os.path.join(synth_dir, f"synth_{i}_sat.jpg")
            mp = os.path.join(synth_dir, f"synth_{i}_mask.png")
            s_img = np.random.randint(60, 150, (512, 512, 3), dtype=np.uint8)
            s_msk = np.zeros((512, 512), dtype=np.uint8)
            cv2.line(s_img, (30, 50 + i * 20), (480, 450 - i * 20), (220, 220, 220), 10)
            cv2.line(s_msk, (30, 50 + i * 20), (480, 450 - i * 20), 255, 10)
            cv2.imwrite(sp, s_img)
            cv2.imwrite(mp, s_msk)
            image_paths.append(sp)
            mask_paths.append(mp)

    # Handle tiny sample datasets gracefully (e.g., 1-3 images)
    if len(image_paths) == 1:
        image_paths = image_paths * 2
        mask_paths = mask_paths * 2

    rng = random.Random(seed)
    indices = list(range(len(image_paths)))
    rng.shuffle(indices)

    num_val = max(1, int(len(image_paths) * val_split))
    val_idx, train_idx = indices[:num_val], indices[num_val:]

    train_ds = DeepGlobeJPGDataset([image_paths[i] for i in train_idx], [mask_paths[i] for i in train_idx], img_size=CFG.IMAGE_SIZE, is_training=True)
    val_ds = DeepGlobeJPGDataset([image_paths[i] for i in val_idx], [mask_paths[i] for i in val_idx], img_size=CFG.IMAGE_SIZE, is_training=False)

    actual_bs = min(batch_size, len(train_ds))
    drop_last = len(train_ds) > actual_bs

    train_loader = DataLoader(train_ds, batch_size=actual_bs, shuffle=True, drop_last=drop_last, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=min(batch_size, len(val_ds)), shuffle=False, drop_last=False, num_workers=0, pin_memory=True)
    print(f"📊 Dataset Ready: {len(train_ds)} Train | {len(val_ds)} Val images (Batch Size: {actual_bs})")
    return train_loader, val_loader, [image_paths[i] for i in val_idx]




In [6]:

# ==============================================================================
# 6. FP32-SAFE MULTI-TASK LOSS SUITE (MASKED ORIENTATION LOSS)
# ==============================================================================
def soft_skeletonize(x: torch.Tensor, iters: int = 3) -> torch.Tensor:
    skel = torch.zeros_like(x)
    curr = x
    for _ in range(iters):
        eroded = -F.max_pool2d(-curr, 3, 1, 1)
        opened = F.max_pool2d(eroded, 3, 1, 1)
        skel = torch.max(skel, F.relu(curr - opened))
        curr = eroded
    return torch.clamp(skel, 0.0, 1.0)

def soft_cldice_loss(pred_logits: torch.Tensor, targets: torch.Tensor, smooth: float = 1e-4) -> torch.Tensor:
    preds = torch.sigmoid(pred_logits.float())
    targets_f = targets.float()
    skel_pred = soft_skeletonize(preds, iters=3)
    skel_target = soft_skeletonize(targets_f, iters=3)
    tprec = ((skel_pred * targets_f).sum(dim=(2, 3)) + smooth) / (skel_pred.sum(dim=(2, 3)) + smooth)
    tsens = ((skel_target * preds).sum(dim=(2, 3)) + smooth) / (skel_target.sum(dim=(2, 3)) + smooth)
    return 1.0 - ((2.0 * tprec * tsens) / (tprec + tsens + smooth)).mean()

def weighted_focal_bce_loss(pred_logits: torch.Tensor, targets: torch.Tensor,
                            pixel_weights: torch.Tensor, alpha: float = 0.75, gamma: float = 2.0) -> torch.Tensor:
    logits_f = pred_logits.float()
    targets_f = targets.float()
    bce = F.binary_cross_entropy_with_logits(logits_f, targets_f, reduction='none')
    p = torch.sigmoid(logits_f)
    p_t = p * targets_f + (1.0 - p) * (1.0 - targets_f)
    alpha_t = alpha * targets_f + (1.0 - alpha) * (1.0 - targets_f)
    focal = alpha_t * ((1.0 - p_t).clamp(1e-6, 1.0) ** gamma) * bce
    return (focal * pixel_weights.float()).mean()

def soft_dice_loss(pred_logits: torch.Tensor, targets: torch.Tensor, smooth: float = 1.0) -> torch.Tensor:
    preds = torch.sigmoid(pred_logits.float())
    targets_f = targets.float()
    inter = (preds * targets_f).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets_f.sum(dim=(2, 3))
    return 1.0 - ((2.0 * inter + smooth) / (union + smooth)).mean()

def boundary_loss(pred_logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    laplacian = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]],
                             dtype=torch.float32, device=pred_logits.device).view(1, 1, 3, 3)
    b_pred = F.conv2d(torch.sigmoid(pred_logits.float()), laplacian, padding=1).abs()
    b_target = F.conv2d(targets.float(), laplacian, padding=1).abs()
    return F.l1_loss(b_pred, b_target)

class CompositeRoadLoss(nn.Module):
    def __init__(self, cfg: PipelineConfig):
        super().__init__()
        self.cfg = cfg

    def forward(self, r_log, o_log, orient_log, r_tgt, o_tgt, orient_tgt, pixel_weights, epoch, total_epochs):
        with torch.amp.autocast('cuda', enabled=False) if torch.cuda.is_available() else nullcontext():
            r_log_f = r_log.float()
            o_log_f = o_log.float()
            orient_log_f = orient_log.float()

            ramp = min(1.0, float(epoch) / max(float(total_epochs), 1.0))
            w_cldice = self.cfg.W_CLDICE_INIT + ramp * (self.cfg.W_CLDICE_FINAL - self.cfg.W_CLDICE_INIT)

            l_focal = weighted_focal_bce_loss(r_log_f, r_tgt, pixel_weights)
            l_dice = soft_dice_loss(r_log_f, r_tgt)
            l_cldice = soft_cldice_loss(r_log_f, r_tgt)
            l_bound = boundary_loss(r_log_f, r_tgt)
            l_occ = F.binary_cross_entropy_with_logits(o_log_f, o_tgt.float())

            # Masked Orientation Loss: ignore_index=0 prevents background class 0 from collapsing road predictions
            orient_tgt_4d = orient_tgt.unsqueeze(1).float() if orient_tgt.dim() == 3 else orient_tgt.float()
            orient_down = F.interpolate(orient_tgt_4d, size=orient_log_f.shape[2:], mode='nearest').long().squeeze(1)
            if (orient_down > 0).any():
                l_orient = F.cross_entropy(orient_log_f, orient_down, ignore_index=0)
            else:
                l_orient = torch.tensor(0.0, device=r_log.device)

            total = (
                self.cfg.W_FOCAL * l_focal +
                self.cfg.W_DICE * l_dice +
                w_cldice * l_cldice +
                self.cfg.W_BOUNDARY * l_bound +
                self.cfg.W_OCC * l_occ +
                self.cfg.W_ORIENT * l_orient
            )
            return total, {
                'loss_total': float(total.item()),
                'l_dice': float(l_dice.item()),
                'l_orient': float(l_orient.item())
            }




In [7]:
# ==============================================================================
# CELL 7: Problem Statement Evaluation Metrics & TTA Threshold Sweeper
# ==============================================================================
def predict_with_tta(model: nn.Module, imgs: torch.Tensor, use_tta: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
    r_log, o_log, _ = model(imgs)
    if not use_tta:
        return torch.sigmoid(r_log), torch.sigmoid(o_log)

    r_prob = torch.sigmoid(r_log)
    o_prob = torch.sigmoid(o_log)

    r_h, o_h, _ = model(torch.flip(imgs, dims=[3]))
    r_prob += torch.flip(torch.sigmoid(r_h), dims=[3])
    o_prob += torch.flip(torch.sigmoid(o_h), dims=[3])

    r_v, o_v, _ = model(torch.flip(imgs, dims=[2]))
    r_prob += torch.flip(torch.sigmoid(r_v), dims=[2])
    o_prob += torch.flip(torch.sigmoid(o_v), dims=[2])

    r_hv, o_hv, _ = model(torch.flip(imgs, dims=[2, 3]))
    r_prob += torch.flip(torch.sigmoid(r_hv), dims=[2, 3])
    o_prob += torch.flip(torch.sigmoid(o_hv), dims=[2, 3])

    return r_prob / 4.0, o_prob / 4.0

def compute_comprehensive_metrics(road_probs: torch.Tensor, road_targets: torch.Tensor,
                                  occ_targets: Optional[torch.Tensor] = None,
                                  threshold: float = 0.5, buffer_px: int = 4) -> Dict[str, float]:
    with torch.no_grad():
        preds = (road_probs > threshold).float()
        targets_f = road_targets.float()

        tp = (preds * targets_f).sum().item()
        fp = (preds * (1.0 - targets_f)).sum().item()
        fn = ((1.0 - preds) * targets_f).sum().item()

        iou = tp / max(tp + fp + fn, 1e-7) if (tp + fp + fn) > 0 else 1.0
        dice = (2.0 * tp) / max(2.0 * tp + fp + fn, 1e-7) if (tp + fp + fn) > 0 else 1.0
        precision = tp / max(tp + fp, 1e-7) if (tp + fp) > 0 else 1.0
        recall = tp / max(tp + fn, 1e-7) if (tp + fn) > 0 else 1.0

        k_size = 2 * buffer_px + 1
        targets_dilated = F.max_pool2d(targets_f, kernel_size=k_size, stride=1, padding=buffer_px)
        preds_dilated = F.max_pool2d(preds, kernel_size=k_size, stride=1, padding=buffer_px)

        relaxed_tp_prec = (preds * targets_dilated).sum().item()
        relaxed_tp_rec = (targets_f * preds_dilated).sum().item()
        r_prec = relaxed_tp_prec / max(preds.sum().item(), 1e-7)
        r_rec = relaxed_tp_rec / max(targets_f.sum().item(), 1e-7)
        relaxed_iou = (r_prec * r_rec) / max(r_prec + r_rec - r_prec * r_rec, 1e-7)

        occ_recall = recall
        if occ_targets is not None:
            occ_road_mask = (targets_f > 0.5) & (occ_targets.float() > 0.5)
            occ_road_total = occ_road_mask.float().sum().item()
            if occ_road_total > 0:
                occ_tp = (preds * occ_road_mask.float()).sum().item()
                occ_recall = occ_tp / occ_road_total

        return {
            'iou': float(iou), 'relaxed_iou': float(relaxed_iou), 'dice': float(dice),
            'precision': float(precision), 'recall': float(recall), 'occlusion_recall': float(occ_recall)
        }

def evaluate_epoch(model: nn.Module, val_loader: DataLoader, device: torch.device, use_tta: bool = False) -> Dict[str, float]:
    model.eval()
    totals = {'iou': 0.0, 'relaxed_iou': 0.0, 'dice': 0.0, 'precision': 0.0, 'recall': 0.0, 'occlusion_recall': 0.0}
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            imgs, r_tgt, o_tgt = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            r_probs, _ = predict_with_tta(model, imgs, use_tta=use_tta)
            m = compute_comprehensive_metrics(r_probs, r_tgt, o_tgt, threshold=0.5, buffer_px=CFG.RELAXED_IOU_BUFFER_PX)
            for k in totals: totals[k] += m[k]
            count += 1
    return {f"val_{k}": totals[k] / max(count, 1) for k in totals}

def find_optimal_threshold(model: nn.Module, val_loader: DataLoader, device: torch.device, use_tta: bool = True) -> float:
    """Sweeps thresholds to find the peak IoU (usually ~0.42 for TTA)."""
    model.eval()
    candidates = [0.35, 0.40, 0.43, 0.46, 0.50]
    scores = {t: 0.0 for t in candidates}
    count = 0

    with torch.no_grad():
        for batch in val_loader:
            imgs, r_tgt, o_tgt = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            r_probs, _ = predict_with_tta(model, imgs, use_tta=use_tta)
            for t in candidates:
                m = compute_comprehensive_metrics(r_probs, r_tgt, o_tgt, threshold=t)
                scores[t] += m['iou']
            count += 1

    best_t = 0.50
    best_iou = -1.0
    for t in candidates:
        scores[t] /= max(count, 1)
        if scores[t] > best_iou:
            best_iou = scores[t]
            best_t = t

    print(f"🎯 Optimal TTA Threshold Auto-Tuned to: {best_t:.2f} (Peak Val IoU: {best_iou:.4f})")
    return best_t

In [8]:

# ==============================================================================
# 8. TOPOLOGICAL SKELETON HEALING (DSU-MST) & GRAPH CRITICALITY ANALYSIS
# ==============================================================================
class DisjointSetUnion:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, i: int) -> int:
        root = i
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[i] != root:
            nxt = self.parent[i]
            self.parent[i] = root
            i = nxt
        return root

    def union(self, a: int, b: int) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

    def connected(self, a: int, b: int) -> bool:
        return self.find(a) == self.find(b)

def heal_skeleton_dsu_bearing(skeleton_bool_hw: np.ndarray, max_gap_px: float = 35.0, max_angle_deg: float = 40.0):
    h, w = skeleton_bool_hw.shape
    skel_u8 = skeleton_bool_hw.astype(np.uint8)
    num_comps, comp_labels, stats, _ = cv2.connectedComponentsWithStats(skel_u8, connectivity=8)

    pre_lcc_px = int(stats[1:, cv2.CC_STAT_AREA].max()) if num_comps > 1 else 0
    if num_comps <= 2:
        return skeleton_bool_hw.copy(), [], 0.0

    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    neigh = cv2.filter2D(skel_u8, cv2.CV_8U, kernel)
    ey, ex = np.where(neigh == 11)
    endpoints = list(zip(ex.tolist(), ey.tolist()))

    if len(endpoints) < 2:
        return skeleton_bool_hw.copy(), [], 0.0

    bearings = []
    for x0, y0 in endpoints:
        visited = {(x0, y0)}
        cx, cy, trace = x0, y0, [(x0, y0)]
        for _ in range(6):
            moved = False
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dx == 0 and dy == 0:
                        continue
                    nx_pt, ny_pt = cx + dx, cy + dy
                    if 0 <= ny_pt < h and 0 <= nx_pt < w and skeleton_bool_hw[ny_pt, nx_pt] and (nx_pt, ny_pt) not in visited:
                        visited.add((nx_pt, ny_pt))
                        cx, cy = nx_pt, ny_pt
                        trace.append((cx, cy))
                        moved = True
                        break
                if moved:
                    break
            if not moved:
                break
        bx, by = float(x0 - trace[-1][0]), float(y0 - trace[-1][1])
        norm = math.hypot(bx, by)
        bearings.append((bx / norm, by / norm) if norm > 1e-5 else (0.0, 0.0))

    cos_thresh = math.cos(math.radians(max_angle_deg))
    dsu = DisjointSetUnion(num_comps + 1)
    candidates = []

    for i in range(len(endpoints)):
        x1, y1 = endpoints[i]
        c1 = int(comp_labels[y1, x1])
        b1 = bearings[i]
        for j in range(i + 1, len(endpoints)):
            x2, y2 = endpoints[j]
            c2 = int(comp_labels[y2, x2])
            if c1 == c2:
                continue
            dx, dy = float(x2 - x1), float(y2 - y1)
            dist = math.hypot(dx, dy)
            if dist > max_gap_px or dist < 1e-4:
                continue
            gv12, gv21 = (dx / dist, dy / dist), (-dx / dist, -dy / dist)
            ok1 = (b1[0] * gv12[0] + b1[1] * gv12[1] >= cos_thresh) if b1 != (0.0, 0.0) else True
            ok2 = (bearings[j][0] * gv21[0] + bearings[j][1] * gv21[1] >= cos_thresh) if bearings[j] != (0.0, 0.0) else True
            if ok1 and ok2:
                candidates.append({'dist': dist, 'pt1': (x1, y1), 'pt2': (x2, y2), 'c1': c1, 'c2': c2})

    candidates.sort(key=lambda c: c['dist'])
    healed_canvas = skel_u8.copy()
    healed_edges = []

    for cand in candidates:
        if not dsu.connected(cand['c1'], cand['c2']):
            dsu.union(cand['c1'], cand['c2'])
            healed_edges.append((cand['pt1'], cand['pt2']))
            cv2.line(healed_canvas, cand['pt1'], cand['pt2'], 1, 1)

    try:
        from skimage.morphology import skeletonize
        healed_bool = skeletonize(healed_canvas > 0).astype(bool)
    except ImportError:
        healed_bool = healed_canvas > 0

    num_after, _, stats_after, _ = cv2.connectedComponentsWithStats(healed_bool.astype(np.uint8), connectivity=8)
    post_lcc_px = int(stats_after[1:, cv2.CC_STAT_AREA].max()) if num_after > 1 else 0
    conn_gain_pct = ((post_lcc_px - pre_lcc_px) / max(pre_lcc_px, 1)) * 100.0
    return healed_bool, healed_edges, float(max(0.0, conn_gain_pct))

def build_spatial_graph(healed_skel: np.ndarray, healed_edges: list, scale_m: float = 5.0) -> nx.Graph:
    G = nx.Graph()
    h, w = healed_skel.shape
    py, px = np.where(healed_skel)
    for r, c in zip(py.tolist(), px.tolist()):
        G.add_node(f"{c}_{r}", x=int(c), y=int(r), degree=0)

    for r, c in zip(py.tolist(), px.tolist()):
        u = f"{c}_{r}"
        for dy in [0, 1]:
            for dx in [-1, 0, 1]:
                if dy == 0 and dx <= 0:
                    continue
                nr, nc = r + dy, c + dx
                if 0 <= nr < h and 0 <= nc < w and healed_skel[nr, nc]:
                    d_m = float(math.hypot(dx, dy) * scale_m)
                    G.add_edge(u, f"{nc}_{nr}", weight=d_m, length=d_m, healed=False)

    for p1, p2 in healed_edges:
        u, v = f"{int(p1[0])}_{int(p1[1])}", f"{int(p2[0])}_{int(p2[1])}"
        for n_id, pt in [(u, p1), (v, p2)]:
            if not G.has_node(n_id):
                G.add_node(n_id, x=int(pt[0]), y=int(pt[1]), degree=0)
        d_m = float(math.hypot(p2[0] - p1[0], p2[1] - p1[1]) * scale_m)
        G.add_edge(u, v, weight=d_m, length=d_m, healed=True)

    for n in G.nodes():
        G.nodes[n]['degree'] = int(G.degree(n))
    return G

def compute_centrality(G: nx.Graph, k: int = 300) -> Tuple[Dict[str, float], np.ndarray, List[Tuple[int, int]]]:
    if len(G) == 0:
        return {}, np.zeros((512, 512), dtype=np.float32), []
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    G_sub = G.subgraph(comps[0]).copy()
    bc = nx.betweenness_centrality(G_sub, k=min(k, len(G_sub)), weight='weight', normalized=True)

    heatmap = np.zeros((512, 512), dtype=np.float32)
    for n in G.nodes():
        score = float(bc.get(n, 0.0))
        G.nodes[n]['centrality'] = score
        x, y = G.nodes[n]['x'], G.nodes[n]['y']
        if 0 <= y < 512 and 0 <= x < 512:
            heatmap[y, x] = score

    heatmap = cv2.dilate(heatmap, np.ones((5, 5), np.float32), iterations=1)
    heatmap = cv2.GaussianBlur(heatmap, (9, 9), sigmaX=2.0)

    intersections = [n for n in G_sub.nodes() if G_sub.degree(n) >= 3] or list(G_sub.nodes())
    top_nodes = sorted(intersections, key=lambda n: bc.get(n, 0.0), reverse=True)[:5]
    gatekeeper_coords = [(G.nodes[n]['x'], G.nodes[n]['y']) for n in top_nodes]
    return bc, heatmap, gatekeeper_coords

def simulate_flood_and_reroute(G: nx.Graph, flood_radius_px: float = 45.0,
                               scale_m: float = 5.0, speed_ms: float = 8.333) -> dict:
    if len(G) < 2:
        return {
            'flood_center': (256, 256), 'flood_radius_px': flood_radius_px,
            'nodes_total': len(G), 'nodes_flooded': 0, 'loss_pct': 0.0,
            'pre_flood_path': [], 'post_flood_path': [],
            'pre_flood_dist_m': 0.0, 'post_flood_dist_m': 0.0,
            'travel_time_increase_s': 0.0, 'detour_factor': 1.0
        }

    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    G_lcc = G.subgraph(comps[0]).copy()
    lcc_nodes = list(G_lcc.nodes())
    src = min(lcc_nodes, key=lambda n: G_lcc.nodes[n]['x'] + G_lcc.nodes[n]['y'])
    tgt = max(lcc_nodes, key=lambda n: G_lcc.nodes[n]['x'] + G_lcc.nodes[n]['y'])

    try:
        pre_path_nodes = nx.shortest_path(G_lcc, src, tgt, weight='weight')
        pre_dist = float(nx.shortest_path_length(G_lcc, src, tgt, weight='weight'))
    except Exception:
        pre_path_nodes, pre_dist = [], 0.0

    if len(pre_path_nodes) >= 5:
        mid_node = pre_path_nodes[len(pre_path_nodes) // 2]
        flood_center = (G_lcc.nodes[mid_node]['x'], G_lcc.nodes[mid_node]['y'])
    else:
        flood_center = (256, 256)

    cx, cy = flood_center
    flooded_set = {
        n for n in G_lcc.nodes()
        if n not in (src, tgt) and math.hypot(G_lcc.nodes[n]['x'] - cx, G_lcc.nodes[n]['y'] - cy) <= flood_radius_px
    }

    G_dam = G_lcc.copy()
    G_dam.remove_nodes_from(flooded_set)

    post_path_nodes, post_dist = [], float('inf')
    if G_dam.has_node(src) and G_dam.has_node(tgt) and nx.has_path(G_dam, src, tgt):
        post_path_nodes = nx.shortest_path(G_dam, src, tgt, weight='weight')
        post_dist = float(nx.shortest_path_length(G_dam, src, tgt, weight='weight'))
    else:
        G_slow = G_lcc.copy()
        for u, v, d in G_slow.edges(data=True):
            if u in flooded_set or v in flooded_set:
                d['weight'] = d.get('length', scale_m) * 4.0
        if nx.has_path(G_slow, src, tgt):
            post_path_nodes = nx.shortest_path(G_slow, src, tgt, weight='weight')
            post_dist = float(nx.shortest_path_length(G_slow, src, tgt, weight='weight'))

    detour = post_dist / pre_dist if (pre_dist > 0 and math.isfinite(post_dist)) else 1.0
    time_inc_s = max(0.0, (post_dist - pre_dist) / speed_ms) if math.isfinite(post_dist) else 0.0

    return {
        'flood_center': flood_center,
        'flood_radius_px': flood_radius_px,
        'nodes_total': len(G_lcc),
        'nodes_flooded': len(flooded_set),
        'loss_pct': (len(flooded_set) / max(len(G_lcc), 1)) * 100.0,
        'pre_flood_path': [(G_lcc.nodes[n]['x'], G_lcc.nodes[n]['y']) for n in pre_path_nodes],
        'post_flood_path': [(G_lcc.nodes[n]['x'], G_lcc.nodes[n]['y']) for n in post_path_nodes],
        'pre_flood_dist_m': pre_dist,
        'post_flood_dist_m': post_dist,
        'travel_time_increase_s': float(time_inc_s),
        'detour_factor': float(detour)
    }

def run_resilience_ablation(G: nx.Graph, steps: int = 10) -> Tuple[List[dict], float, float]:
    if len(G) == 0:
        return [{'pct': 0.0, 'norm_eff': 1.0}], 0.0, 0.0
    comps = sorted(nx.connected_components(G), key=len, reverse=True)
    G_sub = G.subgraph(comps[0]).copy()
    tot = len(G_sub)
    if tot <= 1:
        return [{'pct': 0.0, 'norm_eff': 1.0}], 0.0, 0.0

    nodes = sorted(G_sub.nodes(), key=lambda n: G_sub.nodes[n].get('centrality', 0.0), reverse=True)

    def calc_eff(g: nx.Graph) -> float:
        n = len(g)
        if n <= 1:
            return 0.0
        sample = list(g.nodes())[:min(n, 150)]
        tot_inv = sum(
            1.0 / d for s in sample
            for t, d in nx.single_source_shortest_path_length(g, s).items()
            if t != s and d > 0
        )
        return tot_inv / max(len(sample) * (n - 1), 1)

    e0 = max(calc_eff(G_sub), 1e-6)
    records = [{'pct': 0.0, 'norm_eff': 1.0}]
    for st in range(1, steps + 1):
        rem = (st * tot) // (steps * 2)
        G_a = G_sub.copy()
        G_a.remove_nodes_from(nodes[:rem])
        records.append({
            'pct': float(st * 50 / steps),
            'norm_eff': max(0.0, min(1.0, calc_eff(G_a) / e0))
        })

    y_vals = [r['norm_eff'] for r in records]
    x_vals = [r['pct'] / 50.0 for r in records]
    auc = float(trapz_fn(y_vals, x_vals)) if trapz_fn else float(np.mean(y_vals))
    resilience_index_r = float(records[min(2, len(records) - 1)]['norm_eff'])
    return records, auc, resilience_index_r



In [9]:
# ==============================================================================
# 9. HEADLESS DELIVERABLE EXPORTER (USES TUNED THRESHOLD)
# ==============================================================================
def clean_road_mask(mask_u8: np.ndarray, min_area: int = 35) -> np.ndarray:
    kernel = np.ones((3, 3), np.uint8)
    opened = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=8)
    cleaned = np.zeros_like(mask_u8)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == i] = 255
    return cleaned

def run_image_pipeline(model: nn.Module, image_path: str, out_dir: str, use_tta: bool = True, thresh: float = 0.42) -> Optional[dict]:
    stem = Path(image_path).stem
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None
    img_512 = cv2.resize(img_bgr, (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE))
    img_rgb = cv2.cvtColor(img_512, cv2.COLOR_BGR2RGB)

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)
    rgb_n = (img_rgb.astype(np.float32) / 255.0 - mean) / std
    inp = torch.from_numpy(rgb_n.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        r_prob_t, o_prob_t = predict_with_tta(model, inp, use_tta=use_tta)
        road_prob = r_prob_t.squeeze().cpu().numpy()
        occ_prob = o_prob_t.squeeze().cpu().numpy()

    raw_mask = (road_prob > thresh).astype(np.uint8) * 255
    if raw_mask.sum() == 0 and road_prob.max() > 0.20:
        raw_mask = (road_prob > float(np.percentile(road_prob, 93))).astype(np.uint8) * 255

    road_mask = clean_road_mask(raw_mask, min_area=CFG.MIN_COMPONENT_AREA)
    occ_mask_u8 = (occ_prob > 0.50).astype(np.uint8) * 255

    try:
        from skimage.morphology import skeletonize
        skel = skeletonize(road_mask > 0).astype(bool)
    except ImportError:
        skel = road_mask > 0

    healed_skel, bridges, conn_gain_pct = heal_skeleton_dsu_bearing(
        skel, max_gap_px=CFG.MAX_GAP_PX, max_angle_deg=CFG.MAX_BEARING_ANGLE_DEG
    )
    G = build_spatial_graph(healed_skel, bridges, scale_m=CFG.METERS_PER_PIXEL)
    _, heatmap, gatekeepers = compute_centrality(G, k=CFG.BETWEENNESS_K)
    flood = simulate_flood_and_reroute(G, flood_radius_px=CFG.FLOOD_RADIUS_PX, scale_m=CFG.METERS_PER_PIXEL)
    ablation, auc, res_index_r = run_resilience_ablation(G)

    cv2.imwrite(os.path.join(out_dir, f"{stem}_mask.png"), road_mask)
    cv2.imwrite(os.path.join(out_dir, f"{stem}_occlusion.png"), occ_mask_u8)
    cv2.imwrite(os.path.join(out_dir, f"{stem}_skeleton.png"), healed_skel.astype(np.uint8) * 255)
    nx.write_graphml(G, os.path.join(out_dir, f"{stem}_network.graphml"))

    fig, axes = plt.subplots(2, 2, figsize=(14, 14), facecolor='#0f0f1a')
    for ax in axes.flat:
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='#8888aa')

    axes[0, 0].imshow(img_rgb)
    axes[0, 0].imshow(np.ma.masked_where(road_mask == 0, road_mask), cmap='spring', alpha=0.55)
    for p1, p2 in bridges:
        axes[0, 0].plot([p1[0], p2[0]], [p1[1], p2[1]], color='#00ffff', lw=2.2)
    axes[0, 0].set_title(f"Extracted Roads & {len(bridges)} DSU Bridges (+{conn_gain_pct:.1f}% LCC)", color='white')
    axes[0, 0].axis('off')

    im2 = axes[0, 1].imshow(heatmap, cmap='inferno')
    for gx, gy in gatekeepers:
        axes[0, 1].scatter([gx], [gy], c='#00ffcc', s=70, marker='*', edgecolors='black', zorder=5)
    axes[0, 1].set_title("Betweenness Centrality & Gatekeeper Nodes (★)", color='white')
    plt.colorbar(im2, ax=axes[0, 1], fraction=0.046)
    axes[0, 1].axis('off')

    axes[1, 0].imshow(img_rgb, alpha=0.45)
    for u, v in list(G.edges())[:4000]:
        ux, uy = G.nodes[u]['x'], G.nodes[u]['y']
        vx, vy = G.nodes[v]['x'], G.nodes[v]['y']
        axes[1, 0].plot([ux, vx], [uy, vy], color='#555577', lw=0.7)
    axes[1, 0].add_patch(plt.Circle(flood['flood_center'], flood['flood_radius_px'], color='#ff0055', alpha=0.35))
    if flood['pre_flood_path']:
        axes[1, 0].plot([p[0] for p in flood['pre_flood_path']], [p[1] for p in flood['pre_flood_path']], color='#00d2ff', lw=2.2)
    if flood['post_flood_path']:
        axes[1, 0].plot([p[0] for p in flood['post_flood_path']], [p[1] for p in flood['post_flood_path']], color='#00ff88', lw=2.2, ls='--')
    axes[1, 0].set_title(f"Flood Reroute (Detour: {flood['detour_factor']:.2f}x | +{flood['travel_time_increase_s']:.1f}s)", color='white')
    axes[1, 0].axis('off')

    axes[1, 1].plot([r['pct'] for r in ablation], [r['norm_eff'] for r in ablation], color='#00d2ff', lw=2.2, marker='o')
    axes[1, 1].fill_between([r['pct'] for r in ablation], [r['norm_eff'] for r in ablation], color='#00d2ff', alpha=0.2)
    axes[1, 1].set_title(f"Resilience Index R={res_index_r:.3f} (AUC: {auc:.4f})", color='white')
    axes[1, 1].set_xlabel("% Gatekeeper Nodes Ablated", color='white')
    axes[1, 1].set_ylabel("Normalized Network Efficiency", color='white')

    plt.tight_layout()
    dash_path = os.path.join(out_dir, f"{stem}_dashboard.png")
    plt.savefig(dash_path, dpi=180, facecolor='#0f0f1a')
    plt.close()

    report = {
        'stem': stem, 'nodes': int(len(G)), 'edges': int(G.number_of_edges()),
        'dsu_bridges': int(len(bridges)), 'connectivity_gain_pct': round(conn_gain_pct, 2),
        'resilience_index_R': round(res_index_r, 4), 'resilience_auc': round(auc, 4),
        'flood_detour_factor': round(flood['detour_factor'], 3),
        'travel_time_increase_s': round(flood['travel_time_increase_s'], 1),
        'gatekeeper_nodes': gatekeepers, 'dashboard_path': dash_path
    }
    with open(os.path.join(out_dir, f"{stem}_metrics.json"), 'w') as f:
        json.dump(report, f, indent=2)
    return report

In [10]:
# ==============================================================================
# 10. SELF-CONTAINED TRAINING ENGINE (RESTORED)
# ==============================================================================
def train_model(cfg: PipelineConfig, data_dir: str, weights_path: Optional[str],
                output_dir: str, epochs: int = 4, batch_size: int = 4):
    is_cuda = (torch.cuda.is_available() and DEVICE.type == 'cuda')
    use_amp = is_cuda and cfg.AMP_ENABLED

    model = RoadExtractionModelV3(in_channels=cfg.IN_CHANNELS)
    model, loaded_ckpt = smart_load_checkpoint(model, weights_path)

    train_loader, val_loader, _ = build_dataloaders(
        data_dir, batch_size=batch_size, val_split=cfg.VAL_SPLIT,
        seed=cfg.SEED, pretrained_model=model
    )

    encoder_params, decoder_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'encoder' in name or 'backbone' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': encoder_params, 'lr': cfg.BACKBONE_LR, 'weight_decay': cfg.WEIGHT_DECAY},
        {'params': decoder_params, 'lr': cfg.BASE_LR, 'weight_decay': cfg.WEIGHT_DECAY}
    ], eps=cfg.ADAM_EPS)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1), eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    criterion = CompositeRoadLoss(cfg)

    os.makedirs(output_dir, exist_ok=True)
    best_val_iou = 0.0

    if loaded_ckpt is not None and isinstance(loaded_ckpt, dict):
        best_val_iou = float(loaded_ckpt.get('best_val_iou', 0.0))
        if 'optimizer_state_dict' in loaded_ckpt:
            try:
                optimizer.load_state_dict(loaded_ckpt['optimizer_state_dict'])
                print("✅ Restored Optimizer state!")
            except Exception:
                pass

    init_metrics = evaluate_epoch(model, val_loader, DEVICE, use_tta=False)
    print(f"📌 Baseline Val IoU: {init_metrics['val_iou']:.4f} | Relaxed IoU: {init_metrics['val_relaxed_iou']:.4f} | Dice: {init_metrics['val_dice']:.4f}")
    best_val_iou = max(best_val_iou, init_metrics['val_iou'])

    grad_accum = max(1, cfg.GRAD_ACCUM_STEPS)
    print(f"🚀 Starting Training for {epochs} epochs on {DEVICE} ({len(train_loader)} batches/epoch)...")

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        n_batches = len(train_loader)
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):
            images = batch[0].to(DEVICE, non_blocking=True)
            road_targets = batch[1].to(DEVICE, non_blocking=True)
            occ_targets = batch[2].to(DEVICE, non_blocking=True)
            orient_targets = batch[3].to(DEVICE, non_blocking=True)
            pixel_weights = batch[4].to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=use_amp) if use_amp else nullcontext():
                road_logits, occ_logits, orient_logits = model(images)
                loss, loss_dict = criterion(
                    road_logits, occ_logits, orient_logits,
                    road_targets, occ_targets, orient_targets, pixel_weights,
                    epoch=epoch, total_epochs=epochs
                )
                loss_scaled = loss / grad_accum

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            if use_amp:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if ((step + 1) % grad_accum == 0) or ((step + 1) == n_batches):
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.MAX_GRAD_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.MAX_GRAD_NORM)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item()
            if (step + 1) % max(1, (n_batches // 5)) == 0 or (step + 1) == n_batches:
                print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] | Step [{step+1:04d}/{n_batches:04d}] | Loss: {loss.item():.4f} (Dice: {loss_dict['l_dice']:.4f}, Orient: {loss_dict['l_orient']:.4f})")

        scheduler.step()
        val_m = evaluate_epoch(model, val_loader, DEVICE, use_tta=False)
        elapsed = time.time() - t0

        print(
            f"[EPOCH {epoch+1:02d}/{epochs:02d}] Train Loss: {running_loss/max(n_batches,1):.4f} | "
            f"Val IoU: {val_m['val_iou']:.4f} | Relaxed IoU: {val_m['val_relaxed_iou']:.4f} | "
            f"Val Dice: {val_m['val_dice']:.4f} | Occ-Recall: {val_m['val_occlusion_recall']:.4f} | Time: {elapsed:.1f}s"
        )

        state_payload = {
            'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_iou': max(best_val_iou, val_m['val_iou']),
            'epoch': epoch
        }
        torch.save(state_payload, os.path.join(output_dir, "latest_checkpoint.pth"))

        if val_m['val_iou'] >= best_val_iou:
            best_val_iou = val_m['val_iou']
            best_path = os.path.join(output_dir, "best_model.pth")
            torch.save(state_payload, best_path)
            print(f"  [*] Best Checkpoint Saved -> {best_path} (Val IoU: {best_val_iou:.4f} | Relaxed IoU: {val_m['val_relaxed_iou']:.4f})")

        gc.collect()
        if is_cuda:
            torch.cuda.empty_cache()

    best_ckpt_path = os.path.join(output_dir, "best_model.pth")
    if os.path.exists(best_ckpt_path):
        model, _ = smart_load_checkpoint(model, best_ckpt_path)
    tta_m = evaluate_epoch(model, val_loader, DEVICE, use_tta=True)
    print(f"\n🏆 Final TTA Evaluation | Val IoU: {tta_m['val_iou']:.4f} | Relaxed IoU (4px): {tta_m['val_relaxed_iou']:.4f} | Val Dice: {tta_m['val_dice']:.4f}")
    return model, tta_m

In [11]:
# ==============================================================================
# 11. IN-MEMORY .IPYNB NOTEBOOK EXPORTER (NO EXTERNAL .PY FILE REQUIRED)
# ==============================================================================
def export_in_memory_notebook(out_ipynb_path: str):
    """Generates a clean .ipynb file directly from memory without opening any external .py file."""
    try:
        # Grab current cell source from IPython history if running inside Jupyter/Kaggle, else inspect module
        ip = get_ipython() if 'get_ipython' in globals() else None
        if ip is not None and hasattr(ip, 'user_ns') and '_ih' in ip.user_ns and len(ip.user_ns['_ih']) > 0:
            full_source = ip.user_ns['_ih'][-1]
        else:
            full_source = inspect.getsource(sys.modules[__name__])

        sections = full_source.split("# ==============================================================================\n")
        cells = [{
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# 🛰️ ISRO PS-4: Route Resilience — Occlusion-Robust Road Extraction & Graph Criticality Analysis\n",
                "**Self-Contained Training, Topological Healing (DSU-MST), & Disaster Stress-Testing Pipeline**"
            ]
        }]

        for i in range(1, len(sections), 2):
            header = sections[i].strip()
            body = sections[i + 1] if (i + 1) < len(sections) else ""
            block = f"# ==============================================================================\n# {header}\n# ==============================================================================\n{body}"
            cells.append({
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": block.splitlines(keepends=True)
            })

        nb = {
            "cells": cells,
            "metadata": {
                "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                "language_info": {"name": "python", "version": "3.11.0"}
            },
            "nbformat": 4,
            "nbformat_minor": 5
        }
        with open(out_ipynb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, indent=2)
        print(f"📓 Exported standalone Jupyter Notebook to: {out_ipynb_path}")
    except Exception as e:
        print(f"ℹ️ Notebook export skipped ({e})")

In [12]:
# ==============================================================================
# CELL 12: Master Execution (With Auto-Threshold Integration)
# ==============================================================================
def main():
    print("🚀 Launching ISRO PS-4 Route Resilience Master Pipeline (Tuning Mode)...")

    # 1. Train / Fine-Tune with new clDice emphasis
    train_target_dir = CONFIG_PATHS['train_dir'] or CONFIG_PATHS['sample_dir'] or CONFIG_PATHS['output_dir']
    model, _ = train_model(
        cfg=CFG, data_dir=train_target_dir, weights_path=CONFIG_PATHS['weights'],
        output_dir=CONFIG_PATHS['output_dir'], epochs=CFG.EPOCHS, batch_size=CFG.BATCH_SIZE
    )

    # 2. Run Threshold Sweeper to maximize TTA performance
    _, val_loader, _ = build_dataloaders(train_target_dir, batch_size=CFG.BATCH_SIZE, val_split=CFG.VAL_SPLIT)
    best_thresh = find_optimal_threshold(model, val_loader, DEVICE, use_tta=CFG.USE_TTA)

    # 3. Generate Final Deliverables using optimal threshold
    sample_images = []
    if CONFIG_PATHS['sample_dir'] and os.path.isdir(CONFIG_PATHS['sample_dir']):
        sample_images = sorted([
            p for p in glob.glob(os.path.join(CONFIG_PATHS['sample_dir'], '**/*.jpg'), recursive=True)
            if not p.endswith('_mask.jpg')
        ])
    if not sample_images and CONFIG_PATHS['train_dir']:
        sample_images = sorted(glob.glob(os.path.join(CONFIG_PATHS['train_dir'], '*_sat.jpg')))[:5]

    print(f"\n📊 Generating Phase I, II & III Deliverables on {min(len(sample_images), 13)} image(s)...")
    for img_p in sample_images[:13]:
        # Pass the tuned threshold into the pipeline
        rep = run_image_pipeline(model, img_p, CONFIG_PATHS['output_dir'], use_tta=CFG.USE_TTA, thresh=best_thresh)
        if rep:
            print(
                f"  ✅ {rep['stem']} | Bridges: {rep['dsu_bridges']} (+{rep['connectivity_gain_pct']:.1f}% LCC) | "
                f"Resilience R: {rep['resilience_index_R']:.3f} | Flood Detour: {rep['flood_detour_factor']:.2f}x"
            )

    try:
        export_in_memory_notebook(os.path.join(CONFIG_PATHS['output_dir'], "route_resilience_master.ipynb"))
    except Exception:
        pass
        
    print("\n🎉 Tuning Pipeline Finished! All updated deliverables saved in:", CONFIG_PATHS['output_dir'])

if __name__ == '__main__':
    main()

🚀 Launching ISRO PS-4 Route Resilience Master Pipeline (Tuning Mode)...
📦 Loading checkpoint from: /kaggle/input/datasets/rajitdhakad/0-70-iou-1/0.70iou_1
✅ Smart Checkpoint Loaded: 420/420 layers matched! (Exact: 420, Remapped: 0)
📊 Dataset Ready: 5604 Train | 622 Val images (Batch Size: 4)
✅ Restored Optimizer state!
📌 Baseline Val IoU: 0.6920 | Relaxed IoU: 0.9197 | Dice: 0.8164
🚀 Starting Training for 4 epochs on cuda (1401 batches/epoch)...
  Epoch [01/04] | Step [0280/1401] | Loss: 0.3510 (Dice: 0.2161, Orient: 1.6290)
  Epoch [01/04] | Step [0560/1401] | Loss: 0.4183 (Dice: 0.2538, Orient: 1.7590)
  Epoch [01/04] | Step [0840/1401] | Loss: 0.3108 (Dice: 0.1752, Orient: 1.6436)
  Epoch [01/04] | Step [1120/1401] | Loss: 0.4030 (Dice: 0.2481, Orient: 1.7827)
  Epoch [01/04] | Step [1400/1401] | Loss: 0.3853 (Dice: 0.2111, Orient: 1.8867)
  Epoch [01/04] | Step [1401/1401] | Loss: 0.4341 (Dice: 0.2254, Orient: 2.0532)
[EPOCH 01/04] Train Loss: 0.3784 | Val IoU: 0.6868 | Relaxed IoU

In [13]:
# ==============================================================================
# PHASE 4: SEMI-SUPERVISED PSEUDO-LABELING & FINAL INFERENCE
# ==============================================================================
import os, glob, cv2, torch
import numpy as np

def run_semi_supervised_tuning():
    print("🚀 Launching Semi-Supervised Pseudo-Labeling Pipeline...")
    
    # 1. Setup Paths
    valid_dir = '/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/valid'
    if not os.path.exists(valid_dir):
        valid_dir = '/kaggle/input/deepglobe-road-extraction-dataset/valid'
        
    pseudo_mask_dir = os.path.join(CONFIG_PATHS['output_dir'], 'pseudo_masks_valid')
    os.makedirs(pseudo_mask_dir, exist_ok=True)
    
    # 2. Load the Best Model (prefer newly tuned best_model.pth if available)
    latest_best = os.path.join(CONFIG_PATHS['output_dir'], 'best_model.pth')
    ckpt_to_load = latest_best if os.path.exists(latest_best) else CONFIG_PATHS['weights']
    model = RoadExtractionModelV3(in_channels=CFG.IN_CHANNELS)
    model, loaded_ckpt = smart_load_checkpoint(model, ckpt_to_load)
    
    # 3. Generate High-Quality Pseudo-Masks on unlabelled DeepGlobe Valid set
    unlabelled_imgs = sorted(glob.glob(os.path.join(valid_dir, '*.jpg'))) if os.path.exists(valid_dir) else []
    if not unlabelled_imgs or not CONFIG_PATHS['train_dir']:
        print("⚠️ DeepGlobe 'valid' or 'train' folder not attached. Skipping semi-supervised pass.")
        return
        
    print(f"🛠️ Generating TTA Pseudo-Masks for {len(unlabelled_imgs)} unlabelled DeepGlobe images...")
    pseudo_img_paths, pseudo_mask_paths = [], []
    
    model.eval()
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)
    
    with torch.no_grad():
        for idx, img_p in enumerate(unlabelled_imgs):
            stem = Path(img_p).stem
            m_path = os.path.join(pseudo_mask_dir, f"{stem}_mask.png")
            
            pseudo_img_paths.append(img_p)
            pseudo_mask_paths.append(m_path)
            
            if os.path.exists(m_path):
                continue
            
            img_bgr = cv2.imread(img_p)
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(cv2.resize(img_bgr, (CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)), cv2.COLOR_BGR2RGB)
            rgb_n = (img_rgb.astype(np.float32) / 255.0 - mean) / std
            inp = torch.from_numpy(rgb_n.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
            
            r_probs, _ = predict_with_tta(model, inp, use_tta=True)
            prob = r_probs.squeeze().cpu().numpy()
            
            mask_u8 = (prob > 0.45).astype(np.uint8) * 255
            cv2.imwrite(m_path, mask_u8)
            
            if (idx + 1) % 200 == 0:
                print(f"  ... Generated {idx + 1}/{len(unlabelled_imgs)} pseudo-masks")

    # 4. Combine Labeled Train Data + Pseudo-Labeled Valid Data
    train_sat_files = sorted(glob.glob(os.path.join(CONFIG_PATHS['train_dir'], '**/*_sat.jpg'), recursive=True))
    train_img_paths, train_mask_paths = [], []
    for p in train_sat_files:
        m_png = p.replace('_sat.jpg', '_mask.png')
        if os.path.exists(m_png):
            train_img_paths.append(p)
            train_mask_paths.append(m_png)
            
    combined_images = train_img_paths + pseudo_img_paths
    combined_masks = train_mask_paths + pseudo_mask_paths
    
    print(f"\n🔥 Commencing Semi-Supervised Fine-Tuning on {len(combined_images)} total images!")
    
    CFG.BASE_LR = 2.0e-5
    CFG.BACKBONE_LR = 5.0e-6
    CFG.W_DICE = 0.50
    CFG.W_CLDICE_FINAL = 0.50
    
    # Validate strictly on real ground-truth train split (never validate on pseudo-masks)
    rng = random.Random(CFG.SEED)
    gt_indices = list(range(len(train_img_paths)))
    rng.shuffle(gt_indices)
    num_val = max(1, int(len(train_img_paths) * CFG.VAL_SPLIT))
    val_idx = gt_indices[:num_val]
    train_gt_idx = gt_indices[num_val:]
    
    final_train_imgs = [train_img_paths[i] for i in train_gt_idx] + pseudo_img_paths
    final_train_msks = [train_mask_paths[i] for i in train_gt_idx] + pseudo_mask_paths
    final_val_imgs = [train_img_paths[i] for i in val_idx]
    final_val_msks = [train_mask_paths[i] for i in val_idx]
    
    train_ds = DeepGlobeJPGDataset(final_train_imgs, final_train_msks, is_training=True)
    val_ds = DeepGlobeJPGDataset(final_val_imgs, final_val_msks, is_training=False)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    encoder_params, decoder_params = [], []
    for name, param in model.named_parameters():
        if param.requires_grad:
            if 'encoder' in name or 'backbone' in name:
                encoder_params.append(param)
            else:
                decoder_params.append(param)
            
    optimizer = torch.optim.AdamW([
        {'params': encoder_params, 'lr': CFG.BACKBONE_LR, 'weight_decay': CFG.WEIGHT_DECAY},
        {'params': decoder_params, 'lr': CFG.BASE_LR, 'weight_decay': CFG.WEIGHT_DECAY}
    ], eps=CFG.ADAM_EPS)
    
    scaler = torch.amp.GradScaler('cuda', enabled=CFG.AMP_ENABLED)
    criterion = CompositeRoadLoss(CFG)
    
    best_val_iou = float(loaded_ckpt.get('best_val_iou', 0.70)) if isinstance(loaded_ckpt, dict) else 0.70
    n_batches = len(train_loader)
    
    for epoch in range(3):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            images, r_tgt, o_tgt, orient_tgt, p_weights = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.amp.autocast('cuda', enabled=CFG.AMP_ENABLED):
                r_log, o_log, orient_log = model(images)
                loss, _ = criterion(r_log, o_log, orient_log, r_tgt, o_tgt, orient_tgt, p_weights, epoch=epoch, total_epochs=3)
                loss_scaled = loss / CFG.GRAD_ACCUM_STEPS
                
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
                
            scaler.scale(loss_scaled).backward()
            if ((step + 1) % CFG.GRAD_ACCUM_STEPS == 0) or ((step + 1) == n_batches):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
        val_m = evaluate_epoch(model, val_loader, DEVICE, use_tta=False)
        print(f"[Semi-Supervised Epoch {epoch+1}/3] Val IoU: {val_m['val_iou']:.4f} | Relaxed IoU: {val_m['val_relaxed_iou']:.4f} | Dice: {val_m['val_dice']:.4f}")
        
        if val_m['val_iou'] >= best_val_iou:
            best_val_iou = val_m['val_iou']
            torch.save({
                'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
                'best_val_iou': best_val_iou,
                'epoch': epoch
            }, os.path.join(CONFIG_PATHS['output_dir'], "best_model.pth"))

    print("\n🎯 Finding Optimal Threshold...")
    # Fixed single-value return unpacking (matches Cell 7 definition)
    best_thresh = find_optimal_threshold(model, val_loader, DEVICE, use_tta=True)

    # 5. Final Inference on Blind Sample Data
    print("\n📊 Generating Final Phase I, II & III Deliverables on Sample Images...")
    sample_images = sorted([p for p in glob.glob(os.path.join(CONFIG_PATHS['sample_dir'], '**/*.jpg'), recursive=True) if not p.endswith('_mask.jpg')])
    for img_p in sample_images[:13]:
        rep = run_image_pipeline(model, img_p, CONFIG_PATHS['output_dir'], use_tta=True, thresh=best_thresh)
        if rep:
            print(f"  ✅ {rep['stem']} | Bridges: {rep['dsu_bridges']} (+{rep['connectivity_gain_pct']:.1f}% LCC) | Resilience R: {rep['resilience_index_R']:.3f} | Flood Detour: {rep['flood_detour_factor']:.2f}x")

run_semi_supervised_tuning()

🚀 Launching Semi-Supervised Pseudo-Labeling Pipeline...
📦 Loading checkpoint from: /kaggle/input/datasets/rajitdhakad/0-70-iou-1/0.70iou_1
✅ Smart Checkpoint Loaded: 420/420 layers matched! (Exact: 420, Remapped: 0)
🛠️ Generating TTA Pseudo-Masks for 1243 unlabelled DeepGlobe images...
  ... Generated 200/1243 pseudo-masks
  ... Generated 400/1243 pseudo-masks
  ... Generated 600/1243 pseudo-masks
  ... Generated 800/1243 pseudo-masks
  ... Generated 1000/1243 pseudo-masks
  ... Generated 1200/1243 pseudo-masks

🔥 Commencing Semi-Supervised Fine-Tuning on 7469 total images!
[Semi-Supervised Epoch 1/3] Val IoU: 0.6819 | Relaxed IoU: 0.9170 | Dice: 0.8091
[Semi-Supervised Epoch 2/3] Val IoU: 0.6814 | Relaxed IoU: 0.9166 | Dice: 0.8087
[Semi-Supervised Epoch 3/3] Val IoU: 0.6824 | Relaxed IoU: 0.9164 | Dice: 0.8094

🎯 Finding Optimal Threshold...
🎯 Optimal TTA Threshold Auto-Tuned to: 0.50 (Peak Val IoU: 0.6941)

📊 Generating Final Phase I, II & III Deliverables on Sample Images...
  ✅ 19